In [ ]:
%pip install -q -U "google-genai>=1.34.0"

In [ ]:

import json
from google import genai
from google.genai import types
import os


In [ ]:
GOOGLE_API_KEY = 'xxxxxxxxxxxxxxxxx'
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

client = genai.Client(api_key=GOOGLE_API_KEY)
MODEL_ID = 'gemini-2.5-flash'
jobs = []
folder_name = "folder_name"



for i in range(8, 11):
    json_file_path = f'my-batch-requests-taxonomy-{i}.jsonl'
    # 2. Upload JSONL file to File API.
    print(f"Uploading file: {json_file_path}")
    uploaded_batch_requests = client.files.upload(
        file=f'{folder_name}/{json_file_path}',
        config=types.UploadFileConfig(display_name=json_file_path, mime_type='jsonl')
    )
    print(f"Uploaded file: {uploaded_batch_requests.name}")

    batch_job_from_file = client.batches.create(
    model=MODEL_ID,
    src=uploaded_batch_requests.name,
    config={
        'display_name': f'{json_file_path}-job',
        }
    )
    print(f"Created batch job from file: {batch_job_from_file.name}")
    jobs.append((batch_job_from_file, json_file_path))

In [ ]:
import time

# Poll the job status until it's completed.
# print(len(jobs))
while True:
    for batch_job_obj, json_file_path in jobs:
        batch_job = client.batches.get(name=batch_job_obj.name)
        if batch_job.state.name in ('JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED'):
            print(f"Job {batch_job.name} ended with state: {batch_job.state.name}")
            continue
        print(f"Job not finished. Current state: {batch_job.state.name}. Waiting 30 seconds...")
    time.sleep(30)

#     print(f"Job finished with state: {batch_job.state.name}")
#     if batch_job.state.name == 'JOB_STATE_FAILED':
#         print(f"Error: {batch_job.error}")

In [12]:
for batch_job_obj, file_path in jobs:
    batch_job = client.batches.get(name=batch_job_obj.name)
    if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
        # The output is in another file.

        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("\nDownloading and parsing result file content...")
        file_content_bytes = client.files.download(file=result_file_name)
        file_content = file_content_bytes.decode('utf-8')

        with open(f'{folder_name}/{file_path}_result.jsonl', 'w') as f:
            f.write(file_content)
    else:
        print(f"Job did not succeed. Final state: {batch_job.state.name}")

Results are in file: files/batch-1j1g91kzf69062baytezhndwlp0oahp0e58n

Results are in file: files/batch-9dd1kwzbcdi3izu0bdkiqa19psufc798hz05

Results are in file: files/batch-a3kx42tqbcvpqs9um7tcmccpektvthkinu2o

